# RQ2, Part 2: Real num_reassignments Extraction

Real extraction of reassignment history from JIRA's changelog API, for a random 3,000-issue sample (simple random sampling, not stratified -- see note below). Independent of Part 1 -- can run in a separate tab at the same time.

**Requires `apache_jira_raw.csv` from Part 1** (uploaded or already present in this session).

Output: `num_reassignments_real.csv`

In [1]:
!pip install -q pandas numpy requests statsmodels scipy || pip install -q pandas numpy requests statsmodels scipy --break-system-packages

In [2]:
"""
RQ2 REFINEMENT: Real num_reassignments extraction from JIRA changelog
================================================================================
Fixes a real, previously-open limitation: num_reassignments was named as an
RQ2 predictor in the original synopsis but was never extracted, since the
original JIRA extraction only pulled basic issue fields, not changelog
history.

Real method, confirmed against Atlassian's own REST API documentation: fetch
each issue with `?expand=changelog`, then count how many changelog entries
have field == "assignee" (each one is a real reassignment event).

Run in Colab. Scoped to a random sample of 3,000 issues (not the
full 30,733) to keep runtime reasonable (~20-30 minutes); this comfortably
exceeds RQ2's original minimum sample size target of N=115.

Note: this is a SIMPLE RANDOM SAMPLE (pandas .sample(), no strata).
It is not stratified by project, era, or priority. If a stratified
design is required for the interim report, replace the .sample() call
below with a stratified draw, e.g.:
    df.groupby(['project_name', df['resolution_date'].str[:4].astype(float) >= 2023]) \\
      .apply(lambda g: g.sample(frac=min(1, sample_size/len(df)), random_state=42))
"""

import requests
import time
import os
import pandas as pd
import numpy as np

HEADERS = {"User-Agent": "qm640-capstone"}
np.random.seed(42)


def ensure_jira_data():
    path = "apache_jira_raw.csv"
    if os.path.exists(path):
        return pd.read_csv(path)
    print("apache_jira_raw.csv not found locally -- downloading from your GitHub repo...")
    url = "https://raw.githubusercontent.com/daljeetkaurJohar/qm640-governance-analytics/master/data/raw/apache_jira_raw.csv"
    resp = requests.get(url, headers=HEADERS, timeout=60)
    with open(path, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content)} bytes.")
    return pd.read_csv(path)


def get_real_reassignment_count(issue_key: str) -> int:
    """Real fetch from the live JIRA REST API. Returns the real count of
    assignee-field changes in the issue's changelog, or None on failure."""
    url = f"https://issues.apache.org/jira/rest/api/2/issue/{issue_key}?expand=changelog&fields=summary"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            return None
        data = resp.json()
        histories = data.get("changelog", {}).get("histories", [])
        reassignments = 0
        for history in histories:
            for item in history.get("items", []):
                if item.get("field") == "assignee":
                    reassignments += 1
        return reassignments
    except Exception:
        return None


def run_reassignment_extraction(sample_size: int = 3000, save_every: int = 100):
    df = ensure_jira_data()
    print(f"Loaded {len(df)} total real issues.")

    sample_df = df.sample(n=min(sample_size, len(df)), random_state=42).reset_index(drop=True)
    print(f"Real random sample (not stratified): {len(sample_df)} issues (target: N=115 minimum from synopsis, "
          f"comfortably exceeded)")

    rows = []
    for i, (_, row) in enumerate(sample_df.iterrows()):
        count = get_real_reassignment_count(row["issue_id"])
        rows.append({"issue_id": row["issue_id"], "num_reassignments": count})
        time.sleep(0.2)  # respect the real JIRA API's rate limits

        if (i + 1) % save_every == 0:
            pd.DataFrame(rows).to_csv("num_reassignments_progress.csv", index=False)
            success_so_far = sum(1 for r in rows if r["num_reassignments"] is not None)
            print(f"  [{i+1}/{len(sample_df)}] processed, {success_so_far} real successful fetches so far")

    out = pd.DataFrame(rows)
    out.to_csv("num_reassignments_real.csv", index=False)
    real_counts = out["num_reassignments"].dropna()
    print(f"\n{'='*60}")
    print("EXTRACTION COMPLETE")
    print(f"{'='*60}")
    print(f"Real successful fetches: {len(real_counts)} of {len(out)}")
    if len(real_counts):
        print(f"Real distribution: mean={real_counts.mean():.2f}, median={real_counts.median():.0f}, "
              f"max={real_counts.max():.0f}")
        print(f"Issues with 0 real reassignments: {(real_counts == 0).sum()} "
              f"({(real_counts == 0).mean()*100:.1f}%)")
    print(f"\nSend me: num_reassignments_real.csv")
    return out


if __name__ == "__main__":
    result = run_reassignment_extraction(sample_size=3000)


Loaded 30996 total real issues.
Real stratified sample: 3000 issues (target: N=115 minimum from synopsis, comfortably exceeded)
  [100/3000] processed, 100 real successful fetches so far
  [200/3000] processed, 199 real successful fetches so far
  [300/3000] processed, 299 real successful fetches so far
  [400/3000] processed, 399 real successful fetches so far
  [500/3000] processed, 499 real successful fetches so far
  [600/3000] processed, 599 real successful fetches so far
  [700/3000] processed, 699 real successful fetches so far
  [800/3000] processed, 799 real successful fetches so far
  [900/3000] processed, 899 real successful fetches so far
  [1000/3000] processed, 999 real successful fetches so far
  [1100/3000] processed, 1099 real successful fetches so far
  [1200/3000] processed, 1198 real successful fetches so far
  [1300/3000] processed, 1297 real successful fetches so far
  [1400/3000] processed, 1397 real successful fetches so far
  [1500/3000] processed, 1497 real su